# Bear Classification with timm + NLLLoss (5-class)

This notebook trains a timm backbone (default: `resnet18`) on a single ImageFolder dataset root, does a reproducible 80/20 split, and reports:
- accuracy
- macro-F1
- per-class accuracy
- confusion matrix

It also uses:
- LR scheduling (`ReduceLROnPlateau`)
- early stopping
- optional class balancing with `WeightedRandomSampler`

In [16]:
import os
import kagglehub
from huggingface_hub import login

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HUGGING_FACE_HUB_TOKEN")

login(secret_value_0)

os.environ['DATA_DIR'] = 'data_dir'

print(f"Environment Variable: {os.environ['DATA_DIR']}")


Environment Variable: data_dir


In [33]:
import os
from pathlib import Path
import shutil
import random
import math

train_split = .8
bears = []
train_dir = Path("/kaggle/working/data_dir/train")
val_dir = Path("/kaggle/working/data_dir/val")
k_path = Path("/kaggle/input/datasets/hoturam/bear-dataset")
random.seed(42)

for root, dirs, files in os.walk(k_path):
    #print(f"Root: {root}\n Dirs: {dirs}\n Files: {files}")
    if root[-len("/data"):] == "/data":
        # List of bears in the /data directory
        bears = dirs
    else: 
        bear = root[root.find("/data/")+len("/data/"):]
        if bear in bears:
            bear_img = files
            random.shuffle(bear_img)
            train_cnt = math.ceil(len(bear_img)*train_split)
            # Create directories
            (train_dir / bear).mkdir(parents=True, exist_ok=True)
            (val_dir / bear).mkdir(parents=True, exist_ok=True)
            # Copy images
            k_in_root = Path(root)
            for img_cnt, img in enumerate(bear_img):
                imp = Path(img)
                if img_cnt+1 <= train_cnt:
                    shutil.copy(k_in_root / img, train_dir / bear / img)
                else:
                    shutil.copy(k_in_root / img, val_dir / bear / img)


In [ ]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import shutil
import kagglehub

"""
/home/icode/.cache/kagglehub/datasets/hoturam/bear-dataset/versions/1
/home/icode/.cache/kagglehub/datasets/hoturam/bear-dataset/versions/1/data/black/black1.jpg
path = kagglehub.dataset_download("hoturam/bear-dataset", path= "data/black/black1.jpg")

"""

def split_dataset(source_dir, train_dir, val_dir, test_size=0.2):
    """Split bear dataset into train/val"""
    source = Path(source_dir)
    
    for class_name in ["black", "grizzly", "panda", "polar", "teddy"]:
        class_path = source / "data" / class_name
        images = list(class_path.glob("*.jpg"))  # adjust extension as needed
        
        train_imgs, val_imgs = train_test_split(
            images, 
            test_size=test_size,  # 0.2 = 80/20 split
            random_state=42
        )
        
        # Create directories and copy files
        (Path(train_dir) / class_name).mkdir(parents=True, exist_ok=True)
        (Path(val_dir) / class_name).mkdir(parents=True, exist_ok=True)
        
        for img in train_imgs:
            shutil.copy(img, Path(train_dir) / class_name / img.name)
        for img in val_imgs:
            shutil.copy(img, Path(val_dir) / class_name / img.name)

if not isBear:
    k_path = kagglehub.dataset_download("hoturam/bear-dataset")
	split_dataset(
	    source_dir=k_path,
	    train_dir="data_dir/train",
	    val_dir="data_dir/val",
	    test_size=0.2  # 80/20 split
    )
    isBear = True


In [ ]:
# If timm is not installed in your Kaggle runtime, uncomment:
# !pip -q install timm

import os
import copy
import time
import random
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Config

In [ ]:
# Update these as needed
DATA_DIR = '/kaggle/input/datasets/hoturam/bear-dataset'  # single ImageFolder root
MODEL_NAME = 'resnet18'
PRETRAINED = True

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

VAL_RATIO = 0.2
BALANCE_TRAIN = False  # set True to use WeightedRandomSampler

EPOCHS = 20
LR = 1e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5

SAVE_PATH = 'best_timm_classifier.pt'
SEED = 42

## Training code

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


class TimmClassifier(nn.Module):
    def __init__(self, model_name='resnet18', num_classes=5, pretrained=True):
        super().__init__()
        try:
            self.base_model = timm.create_model(model_name, pretrained=pretrained)
        except Exception as e:
            raise ValueError(f"Error loading model '{model_name}': {e}")

        if hasattr(self.base_model, 'reset_classifier'):
            self.base_model.reset_classifier(num_classes=num_classes)
        else:
            if hasattr(self.base_model, 'fc'):
                in_features = self.base_model.fc.in_features
                self.base_model.fc = nn.Linear(in_features, num_classes)
            elif hasattr(self.base_model, 'classifier'):
                if isinstance(self.base_model.classifier, nn.Linear):
                    in_features = self.base_model.classifier.in_features
                    self.base_model.classifier = nn.Linear(in_features, num_classes)
                elif isinstance(self.base_model.classifier, nn.Sequential):
                    layers = list(self.base_model.classifier.children())
                    linear_idx = None
                    for i in range(len(layers) - 1, -1, -1):
                        if isinstance(layers[i], nn.Linear):
                            linear_idx = i
                            break
                    if linear_idx is None:
                        raise ValueError('Classifier Sequential has no Linear layer to replace.')
                    in_features = layers[linear_idx].in_features
                    layers[linear_idx] = nn.Linear(in_features, num_classes)
                    self.base_model.classifier = nn.Sequential(*layers)
                else:
                    raise ValueError('Unsupported classifier type for this architecture.')
            else:
                raise ValueError('Unknown model architecture: cannot find classifier layer.')

        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        logits = self.base_model(x)
        return self.log_softmax(logits)


def confusion_matrix_torch(y_true, y_pred, num_classes, device=None):
    if device is None:
        device = y_true.device
    cm = torch.zeros((num_classes, num_classes), dtype=torch.int64, device=device)
    for t, p in zip(y_true, y_pred):
        cm[t.long(), p.long()] += 1
    return cm


def per_class_accuracy_from_cm(cm):
    correct = torch.diag(cm).float()
    totals = cm.sum(dim=1).float().clamp_min(1.0)
    return correct / totals


def macro_f1_from_cm(cm):
    tp = torch.diag(cm).float()
    fp = cm.sum(dim=0).float() - tp
    fn = cm.sum(dim=1).float() - tp

    precision = tp / (tp + fp).clamp_min(1e-12)
    recall = tp / (tp + fn).clamp_min(1e-12)
    f1 = 2 * precision * recall / (precision + recall).clamp_min(1e-12)
    return f1.mean().item(), f1


def build_dataloaders_from_single_root(
    data_dir,
    image_size=224,
    batch_size=32,
    num_workers=2,
    val_ratio=0.2,
    seed=42,
    balance_train=False,
):
    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(f"Expected '{data_dir}' to exist.")

    imagenet_mean = (0.485, 0.456, 0.406)
    imagenet_std = (0.229, 0.224, 0.225)

    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(image_size, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    val_tfms = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ])

    full_ds_for_meta = datasets.ImageFolder(data_dir)
    class_names = full_ds_for_meta.classes
    n = len(full_ds_for_meta)
    if n < 10:
        raise ValueError(f'Dataset too small: only {n} images.')

    val_size = max(1, int(round(n * val_ratio)))
    train_size = n - val_size
    if train_size < 1:
        raise ValueError(f'Split invalid: train_size={train_size}, val_size={val_size}')

    g = torch.Generator().manual_seed(seed)
    indices = torch.randperm(n, generator=g).tolist()
    train_idx = indices[:train_size]
    val_idx = indices[train_size:]

    train_base = datasets.ImageFolder(data_dir, transform=train_tfms)
    val_base = datasets.ImageFolder(data_dir, transform=val_tfms)

    train_subset = torch.utils.data.Subset(train_base, train_idx)
    val_subset = torch.utils.data.Subset(val_base, val_idx)

    train_sampler = None
    if balance_train:
        targets = full_ds_for_meta.targets
        train_targets = [targets[i] for i in train_idx]
        num_classes = len(class_names)

        class_counts = torch.zeros(num_classes, dtype=torch.float)
        for t in train_targets:
            class_counts[t] += 1.0
        class_weights = 1.0 / class_counts.clamp_min(1.0)

        sample_weights = torch.tensor([class_weights[t].item() for t in train_targets], dtype=torch.double)
        train_sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True,
        )

    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=(train_sampler is None),
        sampler=train_sampler,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, val_loader, class_names


def run_one_epoch(model, loader, criterion, optimizer, device, num_classes, train=True):
    model.train() if train else model.eval()

    running_loss = 0.0
    running_correct = 0
    total = 0
    epoch_cm = torch.zeros((num_classes, num_classes), dtype=torch.int64, device=device)

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad()

            log_prob = model(images)
            loss = criterion(log_prob, labels)

            if train:
                loss.backward()
                optimizer.step()

            preds = log_prob.argmax(dim=1)
            bs = labels.size(0)

            running_loss += loss.item() * bs
            running_correct += (preds == labels).sum().item()
            total += bs
            epoch_cm += confusion_matrix_torch(labels, preds, num_classes=num_classes, device=device)

    epoch_loss = running_loss / max(total, 1)
    epoch_acc = running_correct / max(total, 1)
    macro_f1, _ = macro_f1_from_cm(epoch_cm)
    per_class_acc = per_class_accuracy_from_cm(epoch_cm).detach().cpu().tolist()

    return epoch_loss, epoch_acc, macro_f1, per_class_acc, epoch_cm.detach().cpu()


def train_model(
    model,
    train_loader,
    val_loader,
    class_names,
    device,
    epochs=20,
    lr=1e-4,
    weight_decay=1e-4,
    save_path='best_timm_classifier.pt',
    early_stopping_patience=5,
):
    num_classes = len(class_names)
    criterion = nn.NLLLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_val_acc = -1.0
    best_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0

    history = {
        'train_loss': [], 'train_acc': [], 'train_macro_f1': [],
        'val_loss': [], 'val_acc': [], 'val_macro_f1': [], 'lr': []
    }

    start = time.time()
    for epoch in range(1, epochs + 1):
        current_lr = optimizer.param_groups[0]['lr']

        train_loss, train_acc, train_macro_f1, _, _ = run_one_epoch(
            model, train_loader, criterion, optimizer, device, num_classes, train=True
        )
        val_loss, val_acc, val_macro_f1, val_pc_acc, val_cm = run_one_epoch(
            model, val_loader, criterion, optimizer, device, num_classes, train=False
        )

        scheduler.step(val_acc)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['train_macro_f1'].append(train_macro_f1)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_macro_f1'].append(val_macro_f1)
        history['lr'].append(current_lr)

        print(
            f"Epoch [{epoch}/{epochs}] lr={current_lr:.2e} "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} train_macro_f1={train_macro_f1:.4f} "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_macro_f1={val_macro_f1:.4f}"
        )

        pc_str = ' | '.join([f"{name}:{acc:.3f}" for name, acc in zip(class_names, val_pc_acc)])
        print(f"  Val per-class acc -> {pc_str}")
        print('  Val confusion matrix (rows=true, cols=pred):')
        print(val_cm.numpy())

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, save_path)
            epochs_without_improvement = 0
            print(f"  -> Saved new best model to: {save_path}")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= early_stopping_patience:
            print(f"  -> Early stopping triggered (no val_acc improvement for {early_stopping_patience} epochs).")
            break

    elapsed = time.time() - start
    print(f"\nTraining complete in {elapsed/60:.1f} min. Best val_acc={best_val_acc:.4f}")

    model.load_state_dict(best_state)
    return model, history, best_val_acc


@torch.no_grad()
def preview_predictions(model, loader, class_names, device, max_items=12, seed=42):
    model.eval()
    rng = random.Random(seed)

    images_all, labels_all = [], []
    for images, labels in loader:
        images_all.append(images)
        labels_all.append(labels)

    if len(images_all) == 0:
        print('No validation samples available for preview.')
        return

    images_all = torch.cat(images_all, dim=0)
    labels_all = torch.cat(labels_all, dim=0)

    n = images_all.size(0)
    k = min(max_items, n)
    sampled_idx = rng.sample(range(n), k=k)

    sampled_images = images_all[sampled_idx].to(device)
    sampled_labels = labels_all[sampled_idx].to(device)

    log_prob = model(sampled_images)
    probs = log_prob.exp()
    preds = probs.argmax(dim=1)

    print('\n=== Validation prediction preview (random samples) ===')
    for i in range(k):
        gt_idx = sampled_labels[i].item()
        pr_idx = preds[i].item()
        conf = probs[i, pr_idx].item()
        print(f"GT={class_names[gt_idx]:<12} | Pred={class_names[pr_idx]:<12} | Conf={conf:.3f}")

## Run training

In [ ]:
set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

train_loader, val_loader, class_names = build_dataloaders_from_single_root(
    data_dir=DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    val_ratio=VAL_RATIO,
    seed=SEED,
    balance_train=BALANCE_TRAIN,
)

num_classes = len(class_names)
print(f'Classes ({num_classes}): {class_names}')
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')
print(f'Approx samples -> train: {len(train_loader.dataset)}, val: {len(val_loader.dataset)}')

model = TimmClassifier(
    model_name=MODEL_NAME,
    num_classes=num_classes,
    pretrained=PRETRAINED,
).to(device)

model, history, best_val_acc = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    class_names=class_names,
    device=device,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    save_path=SAVE_PATH,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
)

preview_predictions(model, val_loader, class_names, device, max_items=12, seed=SEED)
print('\nDone.')

## (Optional) Quick history print

In [ ]:
print('Best val_acc:', best_val_acc)
print('Epochs ran:', len(history['val_acc']))
for i in range(len(history['val_acc'])):
    print(
        f"epoch={i+1:02d} "
        f"train_acc={history['train_acc'][i]:.4f} "
        f"val_acc={history['val_acc'][i]:.4f} "
        f"val_macro_f1={history['val_macro_f1'][i]:.4f} "
        f"lr={history['lr'][i]:.2e}"
    )